In [39]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 120)

# %matplotlib inline

# Data loading

In [43]:
import glob

parquet_files = glob.glob('data/*.parquet')

df_list = []
for file in parquet_files:
    df = pd.read_parquet(file, engine='fastparquet')
    df_list.append(df)

if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
else:
    print("No se encontraron archivos .parquet en el directorio 'sample_data'.")


combined_df.describe()
combined_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 223549 entries, 0 to 223548
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             223549 non-null  object
 1   comment_text   223549 non-null  object
 2   toxic          223549 non-null  int64 
 3   severe_toxic   223549 non-null  int64 
 4   obscene        223549 non-null  int64 
 5   threat         223549 non-null  int64 
 6   insult         223549 non-null  int64 
 7   identity_hate  223549 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 13.6+ MB


# Cleaning and simplification of columns

In [47]:
# ID is not an relevant columns thats really describe some behavior of data.
relevant_columns = combined_df.columns.drop(['id'])
toxic_df = combined_df[relevant_columns]

# Some study
Esta data al ser enfocada a comentarios y clasificada por tipo de comentario. Asumimos que ninguna columna tiene valores `NaN`, incluso la misma columna principal `coment_text`.

In [48]:
toxic_df.shape, toxic_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 223549 entries, 0 to 223548
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   comment_text   223549 non-null  object
 1   toxic          223549 non-null  int64 
 2   severe_toxic   223549 non-null  int64 
 3   obscene        223549 non-null  int64 
 4   threat         223549 non-null  int64 
 5   insult         223549 non-null  int64 
 6   identity_hate  223549 non-null  int64 
dtypes: int64(6), object(1)
memory usage: 11.9+ MB


((223549, 7), None)

## Análisis de las variables booleanas

In [49]:
# Demostrando que ninguna columna tiene valores NaN
possible_NaN_in_data = np.array([
    toxic_df[column].isna().astype(int)
    for column in toxic_df.columns])
print(possible_NaN_in_data.sum())

total_rows = toxic_df.shape

0


Las variables booleanas representan la clasificación de los comentarios.
En esta parte, observamos la gran mayoría de los comentarios en este dataset, no están clasificados.

In [50]:
bool_columns = toxic_df.select_dtypes(include=['number']).columns
clean_clasification_mask = toxic_df[bool_columns].sum(axis=1) == 0
non_clasified_qyt = clean_clasification_mask.mean() * 100
print(f'Non Clisified rows: {non_clasified_qyt:.2f}%, arrond {clean_clasification_mask.sum()} registers')
print(f'Clisified rows: {(100-non_clasified_qyt):.2f}%, arrond {(~clean_clasification_mask).sum()} registers')


Non Clisified rows: 89.95%, arrond 201081 registers
Clisified rows: 10.05%, arrond 22468 registers


Con esta parte, detallamos un poco la proporción los registros clasificados, prestando atención en la columna de `toxic`. Se puede observar que existen clasificaciones cuando `toxic` está desactivado.

In [56]:
resumen = toxic_df.groupby('toxic')[list(bool_columns.drop('toxic'))].agg(['sum', 'mean'])
resumen_pct = resumen.xs('mean', axis=1, level=1) * 100
resumen_qty = resumen.xs('sum', axis=1, level=1)
print(resumen_pct)
print(resumen_qty)


       severe_toxic    obscene    threat     insult  identity_hate
toxic                                                             
0          0.000000   0.290852  0.017313   0.305691       0.060347
1          9.175084  54.021698  3.058361  49.971942       9.329405
       severe_toxic  obscene  threat  insult  identity_hate
toxic                                                      
0                 0      588      35     618            122
1              1962    11552     654   10686           1995


In [ ]:
threat_mask = (toxic_df['threat'] == 1) & (toxic_df['toxic'] == 0)
threat_not_toxic_rows = toxic_df[threat_mask]
comments = threat_not_toxic_rows['comment_text']

14107     == Help me == \n\n Call 911 I feel like I'm go...
17591     Welcome to my talk page! Like the edits that I...
39932     Bảng mạch chính là một bản mạch đóng vai trò l...
45729     hey!!!!!!! thank you for re-editing my contrib...
51650     Regarding your passing \n\nBecause you willful...
74433     I'm not talking about wikipedia, I'm talking a...
78734     "\n\n Treivas, Miller: you think you're safe o...
89017     If you block me for telling you what you are, ...
90619     wtf? \n\nwtf is your problem? why are you bein...
107703    I shalt sever thy head at the neck. \n\nIt wil...
108646    "\n\nNew Award\n\nWell Done!!!\n\n  The IAmThe...
110326    I am going to bash your skull in and urinate a...
113823    I suppose it would be really, really, really h...
128175    LIL LOKO 13 kill a N14K win a prize kill a sur...
131317                       or else i will hurt your mommy
137628    personal attacks \n\ndont tell me what i can a...
140726    If no-one beats me to it, I'll